# Multi-Vintage Mortgage Data Ingestion


Extend the corrected 2019 target-construction workflow across Freddie Mac origination vintages from 2015 through 2022.


## 1. Setup


In [11]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_processing import build_model_data, build_target

DATA_DIR = Path("../data/raw_mortgage")
YEARS = list(range(2015, 2023))

In [12]:
for year in YEARS:
    orig_path = DATA_DIR / f"sample_{year}" / f"sample_orig_{year}.txt"
    perf_path = DATA_DIR / f"sample_{year}" / f"sample_perf_{year}.txt"
    print(year, "| Orig:", orig_path.exists(), "| Perf:", perf_path.exists())


2015 | Orig: True | Perf: True
2016 | Orig: True | Perf: True
2017 | Orig: True | Perf: True
2018 | Orig: True | Perf: True
2019 | Orig: True | Perf: True
2020 | Orig: True | Perf: True
2021 | Orig: True | Perf: True
2022 | Orig: True | Perf: True


## 2. Dataset Schema


In [13]:
orig_columns = [
    "credit_score",
    "first_payment_date",
    "first_time_homebuyer",
    "maturity_date",
    "msa",
    "mi_percent",
    "num_units",
    "occupancy_status",
    "cltv",
    "dti",
    "original_upb",
    "ltv",
    "original_interest_rate",
    "channel",
    "prepayment_penalty",
    "amortization_type",
    "property_state",
    "property_type",
    "postal_code",
    "loan_id",
    "loan_purpose",
    "original_loan_term",
    "num_borrowers",
    "seller_name",
    "super_conforming_flag",
    "pre_harp_loan_id",
    "special_eligibility_program",
    "harp_indicator",
    "property_valuation_method",
    "interest_only_indicator",
    "vantage_score"
]


## 3. Reusable Data Loaders


## 4. Calendar-Based Target Construction


Use elapsed calendar time from the first-payment month rather than Freddie Mac's reported `loan_age`, which can reset later in a loan's history.


### 4.1 Validate Against 2019


In [14]:
target_2019 = build_target(2019, DATA_DIR)
print("Loans:", len(target_2019))
print("Serious delinquencies:", target_2019["seriously_delinquent_24m"].sum())
print("Rate:", target_2019["seriously_delinquent_24m"].mean())


Loans: 49897
Serious delinquencies: 2285
Rate: 0.04579433633284566


**Observation:** The reusable calendar-based target should reproduce the canonical 2019 population of 49,897 loans with 2,285 serious-delinquency events.


## 5. Build Multi-Vintage Modeling Data


### 5.1 Process All Vintages


In [15]:
vintage_datasets = []
vintage_summary = []

for year in YEARS:
    df = build_model_data(year, DATA_DIR)
    vintage_datasets.append(df)
    vintage_summary.append({
        "vintage": year,
        "loans": len(df),
        "serious_delinquencies": int(df["seriously_delinquent_24m"].sum()),
        "delinquency_rate": df["seriously_delinquent_24m"].mean()
    })

vintage_summary = pd.DataFrame(vintage_summary)
vintage_summary


,vintage,loans,serious_delinquencies,delinquency_rate
0,2015,48626,250,0.005141
1,2016,49613,329,0.006631
2,2017,49841,395,0.007925
3,2018,49908,1274,0.025527
4,2019,49897,2285,0.045794
5,2020,49877,791,0.015859
6,2021,49895,445,0.008919
7,2022,49766,882,0.017723


In [16]:
multi_vintage_data = pd.concat(vintage_datasets, ignore_index=True)
print("Shape:", multi_vintage_data.shape)
print("Unique loans:", multi_vintage_data["loan_id"].nunique())
print("Vintages:", sorted(multi_vintage_data["origination_vintage"].unique()))


Shape: (397423, 31)
Unique loans: 397423
Vintages: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]


**Observation:** The corrected 2015–2022 pipeline should produce 397,423 unique mortgages. Vintage-level event rates vary substantially, motivating out-of-time validation rather than a random pooled split.


## 6. Save Processed Dataset


In [17]:
output_path = "../data/processed/model_data_2015_2022.parquet"
multi_vintage_data.to_parquet(output_path, index=False)
print("Saved:", output_path)


Saved: ../data/processed/model_data_2015_2022.parquet


In [18]:
check = pd.read_parquet(output_path)
print("Shape:", check.shape)
print("Unique loans:", check["loan_id"].nunique())
print("Vintages:", sorted(check["origination_vintage"].unique()))
print()
print(check.groupby("origination_vintage")["seriously_delinquent_24m"].agg(["count", "sum", "mean"]))


Shape: (397423, 31)
Unique loans: 397423
Vintages: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]

                     count   sum      mean
origination_vintage                       
2015                 48626   250  0.005141
2016                 49613   329  0.006631
2017                 49841   395  0.007925
2018                 49908  1274  0.025527
2019                 49897  2285  0.045794
2020                 49877   791  0.015859
2021                 49895   445  0.008919
2022                 49766   882  0.017723


In [19]:
%run ../src/make_figures.py

Saved vintage_delinquency_rates.png
Saved out_of_time_performance.png
Saved risk_decile_lift.png
